In [0]:
%sql
CREATE OR REPLACE TABLE adb_caren.silver.calender AS
WITH dates AS (
  SELECT
    date_add('1940-01-01', pos) AS cal_date
  FROM (
    SELECT explode(sequence(0, datediff('2030-12-31', '1940-01-01'))) AS pos
  )
)
SELECT
  -- Key & raw date
  CAST(date_format(cal_date, 'yyyyMMdd') AS INT)        AS date_key,
  cal_date                                               AS date,

  -- Year
  year(cal_date)                                         AS year,
  CASE WHEN month(cal_date) <= 6 THEN 1 ELSE 2 END      AS year_half,

  -- Quarter
  quarter(cal_date)                                      AS quarter,
  CONCAT('Q', quarter(cal_date))                         AS quarter_name,
  CONCAT(year(cal_date), '-Q', quarter(cal_date))        AS year_quarter,

  -- Month
  month(cal_date)                                        AS month_number,
  date_format(cal_date, 'MMMM')                          AS month_name,
  date_format(cal_date, 'MMM')                           AS month_name_short,
  CONCAT(year(cal_date), '-', lpad(month(cal_date), 2, '0')) AS year_month,

  -- Week
  weekofyear(cal_date)                                   AS week_of_year,
  CONCAT(year(cal_date), '-W', lpad(weekofyear(cal_date), 2, '0')) AS year_week,

  -- Day
  day(cal_date)                                          AS day_of_month,
  dayofweek(cal_date)                                    AS day_of_week,      -- 1=Sun … 7=Sat
  dayofyear(cal_date)                                    AS day_of_year,
  date_format(cal_date, 'EEEE')                          AS day_name,
  date_format(cal_date, 'EEE')                           AS day_name_short,

  -- Weekend / Weekday
  CASE WHEN dayofweek(cal_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend,
  CASE WHEN dayofweek(cal_date) IN (1, 7) THEN FALSE ELSE TRUE END AS is_weekday,

  -- Month boundaries
  trunc(cal_date, 'MM')                                  AS first_day_of_month,
  last_day(cal_date)                                     AS last_day_of_month,
  CASE WHEN cal_date = last_day(cal_date) THEN TRUE ELSE FALSE END AS is_last_day_of_month,

  -- Quarter boundaries
  trunc(cal_date, 'QQ')                                  AS first_day_of_quarter,
  last_day(add_months(trunc(cal_date, 'QQ'), 2))         AS last_day_of_quarter,

  -- Year boundaries
  trunc(cal_date, 'YY')                                  AS first_day_of_year,
  make_date(year(cal_date), 12, 31)                      AS last_day_of_year,

  -- Days remaining
  datediff(last_day(cal_date), cal_date)                 AS days_remaining_in_month,
  datediff(make_date(year(cal_date), 12, 31), cal_date)  AS days_remaining_in_year,

  -- Leap year
  CASE
    WHEN (year(cal_date) % 4 = 0 AND year(cal_date) % 100 != 0)
      OR year(cal_date) % 400 = 0
    THEN TRUE ELSE FALSE
  END                                                    AS is_leap_year,

  -- Prior / Next periods
  add_months(cal_date, -12)                              AS same_day_prior_year,
  add_months(cal_date, -1)                               AS same_day_prior_month,

  -- Fiscal year (assuming Apr–Mar fiscal year; adjust if needed)
  CASE WHEN month(cal_date) >= 4 THEN year(cal_date) ELSE year(cal_date) - 1 END AS fiscal_year,
  CASE
    WHEN month(cal_date) >= 4 THEN month(cal_date) - 3
    ELSE month(cal_date) + 9
  END                                                    AS fiscal_month,
  CASE
    WHEN month(cal_date) BETWEEN 4  AND 6  THEN 1
    WHEN month(cal_date) BETWEEN 7  AND 9  THEN 2
    WHEN month(cal_date) BETWEEN 10 AND 12 THEN 3
    ELSE 4
  END                                                    AS fiscal_quarter

FROM dates


In [0]:
%sql
Select * from adb_caren.silver.calender
order by date desc
limit 10;